In [1]:
import pandas as pd
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.0f}'.format)

In [2]:
# network data
df_network = util.process_network_summary()

In [3]:
county_order = ['King','Kitsap','Pierce','Snohomish','Outside Region','Total']

df_network['county'] = pd.Categorical(df_network['county'], ordered=True,
                   categories=county_order)

In [4]:
#| label: system_summary

# network summary: 'VMT','VHT','total_delay'
vmt = df_network['VMT'].sum()
vht = df_network['VHT'].sum()
delay = df_network['total_delay'].sum()

# Get light rail boardings
df_boardings = pd.read_csv(util.output_path / 'transit/daily_boardings_by_agency.csv')
# transit boardings
transit_ridership = df_boardings['boardings'].sum()
#Light rail boardings
df_line_boardings = pd.read_csv(util.output_path / 'transit/transit_line_results.csv')
lr_ridership = df_line_boardings[df_line_boardings['mode']=='r']['boardings'].sum()

# mode share
df_trip = pd.read_csv(util.output_path / 'agg/dash/mode_share_county.csv')
df = df_trip['trexpfac'].sum() # total trips
transit_share = df_trip.loc[df_trip['mode']=="Transit"]['trexpfac'].sum()/df

# emission
df_emissions = pd.read_csv(util.output_path / 'emissions/emissions_summary.csv')
df = df_emissions.loc[(df_emissions['veh_type'].isin(['light','medium','heavy'])) & \
                      (df_emissions['pollutant_name']=="CO2 Equivalent")].copy()
CO2e = df['total_daily_tons'].sum()

run_summary2 = pd.DataFrame({
    'VMT': [vmt],
    'VHT': [vht],
    'Delay': [delay],
    'Transit Boardings': [transit_ridership],
    'Light Rail Boardings': [lr_ridership],
    '% Transit': [transit_share],
    'CO2e': [CO2e]
})

run_summary2.style.\
        format('{:,.0f}', subset=['VMT','VHT','Delay','Transit Boardings','Light Rail Boardings','CO2e']).\
        format('{:.1%}', subset=['% Transit']).\
            hide(axis="index")

VMT,VHT,Delay,Transit Boardings,Light Rail Boardings,% Transit,CO2e
"82,754,059","2,527,062","216,994","517,974","93,305",2.2%,"40,597"


In [5]:
county_order = ['King','Kitsap','Pierce','Snohomish','Outside Region','Total']

df_network['county'] = pd.Categorical(df_network['county'], ordered=True,
                   categories=county_order)

In [6]:
#| label: total_vmt_vht_delay_by_county

df_vmt = df_network.reset_index().groupby('county',observed=True)['VMT'].sum()
df_vht = df_network.reset_index().groupby('county',observed=True)['VHT'].sum()
df_delay = df_network.reset_index().groupby('county',observed=True)['total_delay'].sum()

df = pd.concat([df_vmt,df_vht,df_delay], axis=1)
df = df[df.index!='Outside Region']
df.rename(columns={'total_delay': 'Total Delay Hours'}, inplace=True)

df.loc['Total',] = df.sum()
total_vmt = df.loc['Total','VMT']

df.style.format('{:,.0f}')

,VMT,VHT,Total Delay Hours
county,,,
King,"43,557,666","1,372,331","152,566"
Kitsap,"4,259,699","124,963","2,167"
Pierce,"18,376,345","546,930","30,162"
Snohomish,"16,231,182","476,508","32,097"
Total,"82,424,892","2,520,731","216,992"


In [7]:
#| label: transit
transit = pd.read_csv(util.output_path / 'transit/daily_boardings_by_agency.csv')

df = transit[['agency_name','boardings']].set_index('agency_name').copy()
df.loc['Total',:] = df['boardings'].sum()
df.rename(columns={'boardings': 'Daily Boardings'}, inplace=True)


df.style.format('{:,.0f}')


,Daily Boardings
agency_name,
King County Metro,"293,382"
Sound Transit,"144,035"
Community Transit,"26,522"
Pierce Transit,"24,231"
Kitsap Transit,"15,177"
Washington Ferries,"9,678"
Everett Transit,"4,949"
Total,"517,974"


In [8]:
#| label: mode_share
mode_share = pd.read_csv(util.output_path / 'agg/dash/mode_share_county.csv')

df = mode_share.groupby('mode')['trexpfac'].sum().reset_index().set_index('mode')
df['Mode Share'] = (df['trexpfac']/ (mode_share['trexpfac'].sum()))
df.loc['Total', :] = df.sum(axis=0)
df['Mode Share'] = (df['Mode Share'] * 100).round(1)
df['Mode Share'] = df['Mode Share'].astype(str) + '%'
df['trexpfac'] = df['trexpfac'].apply(lambda x: f"{int(round(x)):,}")
df.rename(columns={'trexpfac': 'Total Person Trips'}, inplace=True)
df

,Total Person Trips,Mode Share
mode,,
Bike,"244,509",1.5%
HOV2,"3,471,874",21.3%
HOV3+,"2,299,536",14.1%
SOV,"7,328,744",44.9%
School Bus,"258,652",1.6%
Transit,"363,009",2.2%
Walk,"2,356,382",14.4%
Total,"16,322,706",100.0%
